# Logistic Regression

Logistic Regression is a **binary classifier** that models the probability of a class membership using the sigmoid function applied to a linear combination of features.

**Dataset:** Wisconsin Breast Cancer — classify tumours as malignant or benign.

**Algorithm:**  
$$
\hat{p} = \sigma(\mathbf{w}^T \mathbf{x} + b) = \frac{1}{1 + e^{-(\mathbf{w}^T \mathbf{x} + b)}}
$$

Predictions: $\hat{y} = 1$ if $\hat{p} \ge 0.5$ else $0$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import sys
sys.path.insert(0, r"/Jana CMOR/2026_Data_Science_and_Machine_Learning/src/rice_ml/supervised_learning")
from rice_ml.supervised_learning.logistic_regression import LogisticRegression
np.random.seed(42)
print("Imports complete")

## Load & Explore the Dataset

The **Wisconsin Breast Cancer** dataset has 569 samples with 30 numeric features derived from digitised images of fine needle aspirates (FNA) of breast masses.

In [ ]:
data = load_breast_cancer()
X, y = data.data, data.target
feature_names = data.feature_names
class_names   = data.target_names

print(f"Samples: {X.shape[0]} | Features: {X.shape[1]} | Classes: {len(class_names)}")
print(f"Class counts — Malignant (0): {np.sum(y==0)} | Benign (1): {np.sum(y==1)}")
print(f"\nFirst 3 feature names: {list(feature_names[:3])}")

## Exploratory Data Analysis

Class distributions and feature correlations give context before fitting. We visualise the most discriminative features.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Class balance pie chart
axes[0].pie([np.sum(y==0), np.sum(y==1)], labels=class_names,
            colors=['#e63946', '#457b9d'], autopct='%1.1f%%', startangle=90,
            wedgeprops=dict(edgecolor='white', linewidth=2))
axes[0].set_title("Class Distribution", fontsize=12, fontweight='bold')

# Feature distributions — mean radius and mean texture
colors = ['#e63946', '#457b9d']
for cls, color, name in zip([0, 1], colors, class_names):
    axes[1].hist(X[y == cls, 0], bins=20, alpha=0.65, color=color, label=name, edgecolor='k', lw=0.3)
axes[1].set_xlabel("Mean Radius (feature 0)", fontsize=11)
axes[1].set_ylabel("Count", fontsize=11)
axes[1].set_title("Mean Radius by Class", fontsize=12, fontweight='bold')
axes[1].legend()
axes[1].grid(True, linestyle='--', alpha=0.5)

# Scatter: mean radius vs mean texture
for cls, color, name in zip([0, 1], colors, class_names):
    mask = y == cls
    axes[2].scatter(X[mask, 0], X[mask, 1], color=color, alpha=0.5, label=name,
                    edgecolors='k', linewidths=0.3, s=30)
axes[2].set_xlabel("Mean Radius", fontsize=11)
axes[2].set_ylabel("Mean Texture", fontsize=11)
axes[2].set_title("Radius vs Texture by Class", fontsize=12, fontweight='bold')
axes[2].legend()
axes[2].grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

## Feature Correlation Heatmap

Understanding feature correlations helps interpret model weights and identify redundant features.

In [ ]:
import numpy as np
# Correlation on a subset of features for readability
subset_idx = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
subset_names = [feature_names[i] for i in subset_idx]
X_sub = X[:, subset_idx]
corr = np.corrcoef(X_sub.T)

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(corr, cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_xticks(range(len(subset_names)))
ax.set_yticks(range(len(subset_names)))
ax.set_xticklabels(subset_names, rotation=45, ha='right', fontsize=8)
ax.set_yticklabels(subset_names, fontsize=8)
for i in range(len(subset_names)):
    for j in range(len(subset_names)):
        ax.text(j, i, f"{corr[i,j]:.2f}", ha='center', va='center', fontsize=7,
                color='white' if abs(corr[i,j]) > 0.6 else 'black')
plt.colorbar(im, ax=ax, label='Pearson Correlation')
ax.set_title("Feature Correlation Matrix (First 10 Features)", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## Preprocessing & Train/Test Split

We standardise features and use an 80/20 stratified split to preserve class proportions in both sets.

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.20, random_state=42, stratify=y)
print(f"Train: {X_train.shape[0]} | Test: {X_test.shape[0]}")
print(f"Train class ratio — Malignant: {np.mean(y_train==0):.3f} | Benign: {np.mean(y_train==1):.3f}")

## Fit the Model

We train with batch gradient descent, minimising binary cross-entropy loss:
$$L = -\frac{1}{n}\sum_{i=1}^{n}\left[y_i \log(\hat{p}_i) + (1-y_i)\log(1-\hat{p}_i)\right]$$

In [ ]:
model = LogisticRegression(learning_rate=0.1, n_iterations=1000)
model.fit(X_train, y_train)
print(f"Training complete. Weight vector shape: {model.weights.shape}")

## Training Loss Curve

The loss curve shows how cross-entropy decreases over gradient descent iterations. A steadily decreasing curve indicates stable convergence.

In [ ]:
# Re-compute loss history during training
epsilon = 1e-15
losses = []
w, b = np.zeros(X_train.shape[1]), 0.0
lr, n = 0.1, X_train.shape[0]
for _ in range(1000):
    z = X_train @ w + b
    p = 1 / (1 + np.exp(-np.clip(z, -500, 500)))
    p_clip = np.clip(p, epsilon, 1 - epsilon)
    losses.append(-np.mean(y_train * np.log(p_clip) + (1 - y_train) * np.log(1 - p_clip)))
    err = p - y_train
    w -= lr * (X_train.T @ err) / n
    b -= lr * np.sum(err) / n

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(losses, color='steelblue', linewidth=2)
axes[0].set_xlabel("Iteration", fontsize=12)
axes[0].set_ylabel("Binary Cross-Entropy Loss", fontsize=12)
axes[0].set_title("Training Loss Curve", fontsize=13, fontweight='bold')
axes[0].grid(True, linestyle='--', alpha=0.5)

# Loss rate-of-change
delta_loss = np.diff(losses)
axes[1].plot(delta_loss, color='tomato', linewidth=1.5)
axes[1].axhline(0, color='k', linestyle='--', lw=0.8)
axes[1].set_xlabel("Iteration", fontsize=12)
axes[1].set_ylabel("ΔLoss per Step", fontsize=12)
axes[1].set_title("Loss Convergence Rate", fontsize=13, fontweight='bold')
axes[1].grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

## Model Evaluation

We evaluate on the held-out test set using accuracy, precision, recall, and F1-score.

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, auc

y_proba = model.predict_proba(X_test)
y_pred  = model.predict(X_test)

train_acc = model.score(X_train, y_train)
test_acc  = model.score(X_test, y_test)
print(f"Train Accuracy: {train_acc:.4f}")
print(f"Test  Accuracy: {test_acc:.4f}")
print("\n--- Classification Report ---")
print(classification_report(y_test, y_pred, target_names=class_names))

## Confusion Matrix & ROC Curve

The **confusion matrix** shows TP, FP, TN, FN counts.  
The **ROC curve** plots True Positive Rate vs False Positive Rate across decision thresholds; AUC near 1.0 is excellent.

In [ ]:
cm = confusion_matrix(y_test, y_pred)
fpr, tpr, _ = roc_curve(y_test, y_proba)
roc_auc = auc(fpr, tpr)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Confusion matrix
im = axes[0].imshow(cm, cmap='Blues')
axes[0].set_xticks([0, 1]); axes[0].set_yticks([0, 1])
axes[0].set_xticklabels(class_names); axes[0].set_yticklabels(class_names)
axes[0].set_xlabel("Predicted", fontsize=12); axes[0].set_ylabel("Actual", fontsize=12)
axes[0].set_title("Confusion Matrix", fontsize=13, fontweight='bold')
for i in range(2):
    for j in range(2):
        axes[0].text(j, i, cm[i, j], ha='center', va='center', fontsize=18,
                     color='white' if cm[i, j] > cm.max() / 2 else 'black')
plt.colorbar(im, ax=axes[0])

# ROC
axes[1].plot(fpr, tpr, color='steelblue', linewidth=2.5, label=f'AUC = {roc_auc:.4f}')
axes[1].plot([0, 1], [0, 1], 'k--', linewidth=1.2, label='Random classifier')
axes[1].fill_between(fpr, tpr, alpha=0.15, color='steelblue')
axes[1].set_xlabel("False Positive Rate", fontsize=12)
axes[1].set_ylabel("True Positive Rate", fontsize=12)
axes[1].set_title("ROC Curve", fontsize=13, fontweight='bold')
axes[1].legend(fontsize=11)
axes[1].grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

## Top Feature Weights

The absolute weight magnitude indicates a feature's influence on the prediction. Large weights (positive or negative) have the most impact.

In [ ]:
weights = model.weights
top_idx = np.argsort(np.abs(weights))[-10:][::-1]
top_names = [feature_names[i] for i in top_idx]
top_vals  = weights[top_idx]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Signed weights for top features
bar_colors = ['#457b9d' if w > 0 else '#e63946' for w in top_vals]
axes[0].barh(top_names, top_vals, color=bar_colors, edgecolor='k', alpha=0.85)
axes[0].axvline(0, color='k', linewidth=0.8)
axes[0].set_xlabel("Learned Weight", fontsize=12)
axes[0].set_title("Top 10 Feature Weights (Signed)", fontsize=13, fontweight='bold')
axes[0].grid(True, axis='x', linestyle='--', alpha=0.5)

# Absolute weights for all features
all_abs = np.abs(weights)
sort_idx = np.argsort(all_abs)
axes[1].barh(range(len(feature_names)), all_abs[sort_idx], color='steelblue', alpha=0.7)
axes[1].set_yticks(range(len(feature_names)))
axes[1].set_yticklabels([feature_names[i] for i in sort_idx], fontsize=7)
axes[1].set_xlabel("|Weight|", fontsize=12)
axes[1].set_title("All Feature Importances (|Weight|)", fontsize=13, fontweight='bold')
axes[1].grid(True, axis='x', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

## Key Takeaways

- Logistic Regression applies a **sigmoid** to a linear model to produce class probabilities.
- The model converges via **gradient descent** minimising binary cross-entropy.
- Features with large absolute weights have the greatest influence on predictions.
- **Feature scaling** is essential — unscaled features cause slow or unstable convergence.
- AUC-ROC > 0.97 indicates excellent discrimination between malignant and benign tumours.